[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/46_muon_solution.ipynb)

# Solution: Muon Optimizer (Simplified)

Reference solution.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

class MyMuon:
    def __init__(self, params, lr=2e-2, beta=0.95, weight_decay=0.0, ns_steps=5, eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta = beta
        self.weight_decay = weight_decay
        self.ns_steps = ns_steps
        self.eps = eps
        self.momentum_buffers = [torch.zeros_like(p) for p in self.params]

    def _orthogonalize(self, g):
        orig_shape = g.shape
        x = g
        if x.ndim > 2:
            x = x.reshape(x.shape[0], -1)

        transposed = False
        if x.shape[0] > x.shape[1]:
            x = x.T
            transposed = True

        x = x / (x.norm() + self.eps)
        for _ in range(self.ns_steps):
            x = 1.5 * x - 0.5 * (x @ (x.T @ x))

        if transposed:
            x = x.T
        return x.reshape(orig_shape)

    def step(self):
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue

                g = p.grad + self.weight_decay * p
                self.momentum_buffers[i] = self.beta * self.momentum_buffers[i] + (1 - self.beta) * g
                update = self.momentum_buffers[i]

                if p.ndim >= 2:
                    update = self._orthogonalize(update)

                p -= self.lr * update

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


In [ ]:
# Demo
torch.manual_seed(0)
w = torch.randn(8, 4, requires_grad=True)
opt = MyMuon([w], lr=0.05)
for i in range(3):
    loss = (w ** 2).mean()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'step={i}, loss={loss.item():.4f}')

In [ ]:
from torch_judge import check
check('muon')